# MECG: Multilingual Evidence-Grounded Complaint Generation
## Notebook 07: Qwen3-VL QLoRA Fine-Tuning

**Goal**: Fine-tune `Qwen/Qwen3-VL-4B-Instruct` using parameter-efficient QLoRA (4-bit NF4 quantization) to generate natural language product complaints conditioned on multimodal evidence.

**Kaggle GPU Environment Optimization**:
- Target GPU: Kaggle T4 (16 GB VRAM).
- Quantization: 4-bit NF4 (`bitsandbytes`).
- LoRA rank: $r=16$, $\alpha=32$, targeting `q_proj, k_proj, v_proj, o_proj`.
- Memory Savings: Gradient checkpointing enabled, batch size = 1, gradient accumulation = 8.

In [ ]:
# Environment & Hardware Verification
import os
import sys
from pathlib import Path

CURRENT_DIR = Path.cwd()
if str(CURRENT_DIR) not in sys.path:
    sys.path.append(str(CURRENT_DIR))
if str(CURRENT_DIR.parent) not in sys.path:
    sys.path.append(str(CURRENT_DIR.parent))

import torch
print(f'PyTorch Version : {torch.__version__}')
print(f'CUDA Available  : {torch.cuda.is_available()}')
if torch.cuda.is_available():
    print(f'Device Name     : {torch.cuda.get_device_name(0)}')
    print(f'VRAM Total      : {torch.cuda.get_device_properties(0).total_memory / (1024**3):.2f} GB')

### 1. Compile Multimodal Training Samples
Compiles multimodal samples from `ComVID_cleaned.csv`, keyframes, transcripts, and targets.

In [ ]:
from training.prepare_generation_dataset import prepare_datasets

CLEANED_CSV = Path('data/processed/ComVID_cleaned.csv')
if not CLEANED_CSV.exists() and Path('/kaggle/input/comvid-dataset/ComVID_cleaned.csv').exists():
    CLEANED_CSV = Path('/kaggle/input/comvid-dataset/ComVID_cleaned.csv')

prepare_datasets(
    cleaned_csv_path=str(CLEANED_CSV),
    transcripts_path='data/processed/transcripts.jsonl',
    frames_dir='data/processed/frames',
    output_dir='data/processed'
)
print('Training and validation JSONL datasets prepared successfully!')

### 2. Launch QLoRA Fine-Tuning
Runs `train_qlora` using PEFT with 4-bit NF4 quantization.

In [ ]:
from training.train_qwen3vl_qlora import train_qlora

output_adapter_dir = 'outputs/checkpoints/qwen3vl_complaint_lora'

print('Starting QLoRA Fine-tuning on Qwen3-VL...')
train_qlora(
    train_jsonl='data/processed/generation_train.jsonl',
    val_jsonl='data/processed/generation_val.jsonl',
    output_dir=output_adapter_dir,
    base_model_id='Qwen/Qwen3-VL-4B-Instruct',
    epochs=3,
    lr=1e-4,
    lora_r=16,
    lora_alpha=32,
    dry_run=False
)
print(f'LoRA Adapter saved to: {output_adapter_dir}')

### Summary & Next Steps
- Qwen3-VL QLoRA weights fine-tuned and saved.
- Proceed to **Notebook 08: Full Pipeline Demo** for unified end-to-end complaint generation.